In [1]:
import pandas as pd
import random
import json

# Predicates and entities for generation
predicates = [
    "dbo:birthPlace", "dbo:deathPlace", "dbo:occupation",
    "dbo:director", "dbo:starring", "dbo:country"
]

entities = [
    "dbo:Berlin", "dbo:Paris", "dbo:London",
    "dbo:Tokyo", "dbo:NewYork", "dbo:Germany", "dbo:France"
]

extra_relations = [
    "dbo:spouse", "dbo:parent", "dbo:author",
    "dbo:influencedBy", "dbo:team", "dbo:genre"
]

def generate_anchor():
    P = random.choice(predicates)
    O = random.choice(entities)
    return f"""
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?x WHERE {{
    ?x {P} {O} .
}}
""", P, O

def generate_positive(anchor):
    # Minor rewrite
    q = anchor.replace("?x", "?person")
    q = q.replace(" .", " ;")
    return q

def negative_structural():
    # ALWAYS rdflib-compatible UNION
    P = random.choice(predicates)
    O = random.choice(entities)
    return f"""
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?y WHERE {{
    {{
        ?y {P} {O} .
        OPTIONAL {{ ?y dbo:occupation ?job . }}
    }}
    UNION
    {{
        ?y dbo:birthPlace dbo:London .
    }}
}}
"""

def negative_semantic(anchor_pred, anchor_obj):
    # Ensure no predicate or object collision
    P = random.choice([p for p in predicates if p != anchor_pred])
    O = random.choice([e for e in entities if e != anchor_obj])
    extra = random.choice(extra_relations)

    return f"""
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?z WHERE {{
    ?z {P} {O} .
    ?z {extra} dbo:Berlin .
}}
"""

rows = []
n = 10000  # 10k dataset

for i in range(n):
    anc, ancP, ancO = generate_anchor()
    pos = [generate_positive(anc)]
    neg0 = negative_structural()
    neg1 = negative_semantic(ancP, ancO)

    rows.append({
        "id": i,
        "anchor_query": anc,
        "positive_queries": json.dumps(pos),
        "negative_queries": json.dumps([neg0, neg1])
    })

df = pd.DataFrame(rows)
df.to_csv("sample_sparql_queries.csv", index=False)

"Regenerated sample_sparql_queries.csv successfully"


'Regenerated sample_sparql_queries.csv successfully'